In [1]:

import pandas as pd
from pathlib import Path

# Find the Titanic dataset file in Kaggle
dataset_path = Path("/kaggle/input")
file_path = next(dataset_path.rglob("train.csv"))

# Load the raw dataset
df = pd.read_csv(file_path)

# Display basic information
print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())


Dataset loaded successfully!
Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:

# Check data types, missing values, and duplicates

print("=== Dataset Information ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Rows ===")
print("Duplicates:", df.duplicated().sum())

print("\n=== Target Distribution ===")
print(df["Survived"].value_counts())



=== Dataset Information ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

=== Missing Values ===
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Ca

In [3]:

# Select the features and target for binary classification

selected_features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = df[selected_features].copy()
y = df["Survived"].copy()

print("Selected Features:", selected_features)
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

display(X.head())



Selected Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Features Shape: (891, 7)
Target Shape: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [4]:

from sklearn.model_selection import train_test_split

# Split data before preprocessing to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

print("\nTraining Class Distribution:")
print(y_train.value_counts())

print("\nTesting Class Distribution:")
print(y_test.value_counts())


Training Features: (712, 7)
Testing Features: (179, 7)
Training Target: (712,)
Testing Target: (179,)

Training Class Distribution:
Survived
0    439
1    273
Name: count, dtype: int64

Testing Class Distribution:
Survived
0    110
1     69
Name: count, dtype: int64


In [5]:

# Handle missing values using training data only

# Calculate replacement values from training data
age_median = X_train["Age"].median()
embarked_mode = X_train["Embarked"].mode()[0]

# Fill missing values in training and testing data
X_train = X_train.copy()
X_test = X_test.copy()

X_train["Age"] = X_train["Age"].fillna(age_median)
X_test["Age"] = X_test["Age"].fillna(age_median)

X_train["Embarked"] = X_train["Embarked"].fillna(embarked_mode)
X_test["Embarked"] = X_test["Embarked"].fillna(embarked_mode)

# Verify missing values after cleaning
print("Age Median:", age_median)
print("Embarked Mode:", embarked_mode)

print("\nMissing Values in Training Data:")
print(X_train.isnull().sum())

print("\nMissing Values in Testing Data:")
print(X_test.isnull().sum())


Age Median: 28.5
Embarked Mode: S

Missing Values in Training Data:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

Missing Values in Testing Data:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [6]:

# Convert categorical features into numeric columns

categorical_columns = ["Sex", "Embarked"]

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    dtype=int
)

# Ensure the test data has exactly the same columns as training
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("\nEncoded Feature Names:")
print(X_train.columns.tolist())

print("\nFirst 5 Encoded Rows:")
display(X_train.head())


Training Shape: (712, 10)
Testing Shape: (179, 10)

Encoded Feature Names:
['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S']

First 5 Encoded Rows:


,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
692,3,28.5,0,0,56.4958,0,1,0,0,1
481,2,28.5,0,0,0.0000,0,1,0,0,1
527,1,28.5,0,0,221.7792,0,1,0,0,1
855,3,18.0,0,1,9.3500,1,0,0,0,1
801,2,31.0,1,1,26.2500,1,0,0,0,1


In [7]:

from sklearn.preprocessing import StandardScaler
import pandas as pd

# Scale features using training data only
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)

print("\nFirst 5 Scaled Training Rows:")
display(X_train_scaled.head())

print("\nMissing Values After Scaling:")
print("Training:", X_train_scaled.isnull().sum().sum())
print("Testing:", X_test_scaled.isnull().sum().sum())


Scaled Training Shape: (712, 10)
Scaled Testing Shape: (179, 10)

First 5 Scaled Training Rows:


,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
692,0.829568,-0.081135,-0.465084,-0.466183,0.513812,-0.742427,0.742427,-0.492527,-0.289333,0.611978
481,-0.370945,-0.081135,-0.465084,-0.466183,-0.662563,-0.742427,0.742427,-0.492527,-0.289333,0.611978
527,-1.571457,-0.081135,-0.465084,-0.466183,3.955399,-0.742427,0.742427,-0.492527,-0.289333,0.611978
855,0.829568,-0.887827,-0.465084,0.727782,-0.467874,1.346933,-1.346933,-0.492527,-0.289333,0.611978
801,-0.370945,0.110934,0.478335,0.727782,-0.115977,1.346933,-1.346933,-0.492527,-0.289333,0.611978



Missing Values After Scaling:
Training: 0
Testing: 0


In [8]:

# Save processed training and testing data

train_data = pd.DataFrame(X_train_scaled, columns=X_train.columns)
test_data = pd.DataFrame(X_test_scaled, columns=X_test.columns)

train_data["Survived"] = y_train.to_numpy()
test_data["Survived"] = y_test.to_numpy()

train_data.to_csv("/kaggle/working/titanic_train_processed.csv", index=False)
test_data.to_csv("/kaggle/working/titanic_test_processed.csv", index=False)

print("Processed datasets saved successfully!")
print("Training:", train_data.shape)
print("Testing:", test_data.shape)


Processed datasets saved successfully!
Training: (712, 11)
Testing: (179, 11)
